In [7]:
# -*- coding: utf-8 -*-
"""팀 10 학생 평가기 — Colab에서 이 파일 전체가 하나의 코드 셀로 실행됩니다."""

# =====================================================================================
# 사용 방법 — Colab 실행 전에 반드시 확인하세요.
# =====================================================================================
# 1) /content/question 폴더에는 운영진 문항·정답 기준 JSON을 정확히 1개 둡니다.
# 2) /content/terms 폴더에는 카카오 약관 4종을 PDF 또는 TXT로 정확히 4개 둡니다.
# 3) /content/blind 폴더에는 blind01.json~blind05.json 형식의 후보 답변 JSON 5개를 둡니다.
# 4) 후보 JSON 최상위의 어떤 변수든 값이 BLINDxx이면 그 ID를 우선 사용합니다.
#    내부에 BLINDxx 값이 없으면 파일명 blindxx.json에서 ID를 추출합니다.
# 5) 후보들은 BLIND 번호 오름차순으로 정렬되며 최종 eval_10.json에도 같은 ID를 기록합니다.
# 6) Colab 왼쪽 열쇠(Secrets)에 GEMINI_API_KEY를 등록하고 노트북 액세스를 켭니다.
#    API 키를 이 코드나 JSON 파일에 직접 작성하지 마세요.
# 7) 연습으로 후보 수를 줄일 때만 STRICT_FINAL_VALIDATION=False로 변경합니다.
#    실제 최종 실행 전에는 반드시 True로 되돌리고 후보 5개를 모두 준비합니다.
# 8) 문항 수는 코드에 고정되어 있지 않습니다. question의 10~50문항을 동적으로 읽고
#    각 후보의 전체 QID가 question의 전체 QID와 같은지 검사합니다.
# 9) question에는 JSON 1개, terms에는 약관 4개, blind에는 후보 JSON 5개만 둡니다.
# 10) 실행이 끝나면 /content/eval_10.json을 내려받고 다음을 확인합니다.
#    - 최상위 키는 results 하나뿐이며 결과가 정확히 5개인가
#    - BLIND01~BLIND05가 중복 없이 있고 모두 status=completed인가
#    - total이 0~100 숫자이며 rank, real, schema_version 등이 없는가
# 11) partial 또는 failed가 하나라도 있으면 제출 파일 전체가 순위 산정에서 제외될 수 있습니다.
#    체크포인트가 있으므로 오류를 고친 뒤 재실행하면 완료된 동일 답변 판정을 재사용합니다.
# =====================================================================================

# Colab 기본 환경에 없는 공식 Gemini SDK와 한국어 형태소 분석기를 설치합니다.
import sys
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "google-genai", "pydantic", "kiwipiepy", "pypdf"],
    check=True,
)

import json
import os
import re
import time
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Literal

from google import genai
from google.genai import types
from kiwipiepy import Kiwi
from pypdf import PdfReader
from pydantic import BaseModel, Field


# =====================================================================================
# 1. 사용자가 확인할 설정
# =====================================================================================

TEAM_NUMBER = 10
GEMINI_MODEL = "gemini-3.5-flash"

# Colab 입력 폴더를 문항 기준과 후보 답변으로 분리합니다.
ROOT_DIR = Path("/content") if Path("/content").exists() else Path.cwd()
QUESTION_DIR = ROOT_DIR / "question"
TERMS_DIR = ROOT_DIR / "terms"
BLIND_DIR = ROOT_DIR / "blind"
OUTPUT_PATH = ROOT_DIR / f"eval_{TEAM_NUMBER}.json"
CHECKPOINT_PATH = ROOT_DIR / f".eval_{TEAM_NUMBER}_checkpoint.json"
TERMS_CACHE_PATH = ROOT_DIR / ".terms_index_cache.json"

# 공개 개발 중 후보 수가 적으면 False로 바꿀 수 있습니다. 최종 제출 때는 반드시 True입니다.
STRICT_FINAL_VALIDATION = True
EXPECTED_BLIND_COUNT = 5

MAX_API_RETRIES = 3
RETRY_BASE_SECONDS = 2.0
MAX_TERM_CHARS_PER_ARTICLE = 3500

# 최종 100점 배점: MRR 10 + 키팩트 F1 20 + Gemini 평가 70
MRR_WEIGHT = 10.0
F1_WEIGHT = 20.0
LLM_WEIGHT = 70.0

# LLM 100점 내부 비중: 공식 평가 기준과 동일한 네 축입니다.
LLM_AXIS_WEIGHTS = {
    "correctness": 0.40,
    "grounding": 0.25,
    "completeness": 0.20,
    "clarity": 0.15,
}

assert abs(sum(LLM_AXIS_WEIGHTS.values()) - 1.0) < 1e-12
assert abs(MRR_WEIGHT + F1_WEIGHT + LLM_WEIGHT - 100.0) < 1e-12


# =====================================================================================
# 2. Gemini 구조화 출력 스키마
# =====================================================================================

class LLMJudgment(BaseModel):
    """한 문항의 답변에 대한 Gemini 축별 0~10 판정입니다."""

    judgeable: bool
    correctness: float = Field(ge=0, le=10)
    grounding: float = Field(ge=0, le=10)
    completeness: float = Field(ge=0, le=10)
    clarity: float = Field(ge=0, le=10)
    reason: str = Field(min_length=1, max_length=500)


# =====================================================================================
# 3. 입력 파일과 스키마 검증
# =====================================================================================

def load_json(path: Path) -> dict:
    """UTF-8 JSON 객체를 읽고 최상위 타입을 검증합니다."""
    with path.open("r", encoding="utf-8") as file:
        data = json.load(file)
    if not isinstance(data, dict):
        raise ValueError(f"{path.name}: 최상위 JSON은 객체여야 합니다.")
    return data


def find_gold_path(question_dir: Path) -> Path:
    """question 폴더에 있는 유일한 JSON 파일을 문항·정답 기준으로 반환합니다."""
    if not question_dir.is_dir():
        raise FileNotFoundError(f"문항 폴더가 없습니다: {question_dir}")
    files = sorted(question_dir.glob("*.json"))
    if len(files) != 1:
        raise ValueError(f"{question_dir}에는 JSON이 정확히 1개 있어야 하지만 {len(files)}개입니다: {[p.name for p in files]}")
    return files[0]


BLIND_VALUE_RE = re.compile(r"^blind[_-]?(\d+)$", re.IGNORECASE)
BLIND_FILENAME_RE = re.compile(r"blind[_-]?(\d+)", re.IGNORECASE)

def extract_blind_number_from_filename(stem: str) -> int | None:
    """answers_private_BLIND01처럼 파일명 중간에 있는 BLINDxx를 찾습니다."""
    numbers = {int(match.group(1)) for match in BLIND_FILENAME_RE.finditer(stem)}
    numbers.discard(0)
    return next(iter(numbers)) if len(numbers) == 1 else None


def normalize_blind_number(value: object) -> int | None:
    """BLIND01, blind_1 같은 값을 양의 블라인드 번호로 정규화합니다."""
    if not isinstance(value, str):
        return None
    match = BLIND_VALUE_RE.fullmatch(value.strip())
    if match is None:
        return None
    number = int(match.group(1))
    return number if number > 0 else None


def extract_blind_number(data: dict, path: Path) -> int:
    """최상위 내부 BLINDxx 값을 우선하고, 없으면 파일명에서 추출합니다."""
    internal_numbers = {
        number
        for value in data.values()
        if (number := normalize_blind_number(value)) is not None
    }
    if len(internal_numbers) > 1:
        raise ValueError(f"{path.name}: 서로 다른 내부 BLIND ID가 있습니다: {sorted(internal_numbers)}")
    if internal_numbers:
        return next(iter(internal_numbers))
    filename_number = extract_blind_number_from_filename(path.stem)
    if filename_number is not None:
        return filename_number
    raise ValueError(f"{path.name}: 내부 값 또는 파일명에서 BLINDxx를 찾을 수 없습니다.")


def discover_candidate_files(blind_dir: Path) -> list[tuple[int, Path, dict]]:
    """answers 포맷 JSON의 BLIND ID를 찾아 번호 순으로 반환합니다."""
    discovered: list[tuple[int, Path, dict]] = []
    if not blind_dir.is_dir():
        raise FileNotFoundError(f"후보 폴더가 없습니다: {blind_dir}")
    for path in sorted(blind_dir.glob("*.json")):
        data = load_json(path)
        if "answers" not in data:
            raise ValueError(f"{path.name}: blind 폴더의 JSON에는 answers가 필요합니다.")
        discovered.append((extract_blind_number(data, path), path, data))

    if not discovered:
        raise FileNotFoundError(f"{blind_dir}에서 answers 포맷의 후보 JSON을 찾지 못했습니다.")
    discovered.sort(key=lambda item: (item[0], item[1].name))
    blind_numbers = [blind_number for blind_number, _, _ in discovered]
    if len(blind_numbers) != len(set(blind_numbers)):
        raise ValueError(f"후보 BLIND ID가 중복되었습니다: {blind_numbers}")
    if STRICT_FINAL_VALIDATION and len(discovered) != EXPECTED_BLIND_COUNT:
        raise ValueError(f"최종 실행에는 후보 {EXPECTED_BLIND_COUNT}개가 필요하지만 {len(discovered)}개를 찾았습니다.")
    if STRICT_FINAL_VALIDATION and blind_numbers != list(range(1, EXPECTED_BLIND_COUNT + 1)):
        raise ValueError(f"최종 BLIND ID는 1~{EXPECTED_BLIND_COUNT}여야 하지만 {blind_numbers}입니다.")
    return discovered


def validate_gold(data: dict) -> list[dict]:
    """동적 골드 문항 목록과 필수 정답 필드를 검증합니다."""
    questions = data.get("questions")
    if not isinstance(questions, list) or not 10 <= len(questions) <= 50:
        raise ValueError("문항·정답 기준의 questions는 10~50개 배열이어야 합니다.")

    seen_ids: set[str] = set()
    for item in questions:
        if not isinstance(item, dict):
            raise ValueError("각 골드 문항은 객체여야 합니다.")
        qid = item.get("id")
        if not isinstance(qid, str) or not qid:
            raise ValueError("각 골드 문항에는 문자열 id가 필요합니다.")
        if qid in seen_ids:
            raise ValueError(f"골드 문항 id가 중복되었습니다: {qid}")
        seen_ids.add(qid)
        if not isinstance(item.get("question"), str) or not item["question"].strip():
            raise ValueError(f"{qid}: question이 비어 있습니다.")
        if not isinstance(item.get("gold_articles"), list) or not item["gold_articles"]:
            raise ValueError(f"{qid}: gold_articles가 비어 있습니다.")
        if not isinstance(item.get("key_facts"), list) or not item["key_facts"]:
            raise ValueError(f"{qid}: key_facts가 비어 있습니다.")
    return questions


def normalize_article(value: object) -> int:
    """정수 또는 '제3조' 같은 표현에서 조번호를 정수로 추출합니다."""
    if isinstance(value, bool):
        raise ValueError("bool은 조번호가 될 수 없습니다.")
    if isinstance(value, int):
        return value
    match = re.search(r"\d+", str(value))
    if match is None:
        raise ValueError(f"조번호를 해석할 수 없습니다: {value!r}")
    return int(match.group(0))

MAX_RETRIEVED = 4


def coerce_evidence(evidence: object) -> list[object] | None:
    """[문서명, 조번호] 형태만 통과시키고 해석 불가한 항목은 버립니다."""
    if not isinstance(evidence, (list, tuple)) or len(evidence) != 2:
        return None
    doc = evidence[0]
    if not isinstance(doc, str) or not doc.strip():
        return None
    try:
        return [doc.strip(), normalize_article(evidence[1])]
    except ValueError:
        return None


def validate_candidate(data: dict, gold_ids: list[str], filename: str) -> dict[str, dict]:
    """계약 위반은 감점 대상으로 흡수하고, 파싱 불가한 경우에만 예외를 냅니다."""
    answers = data.get("answers")
    if not isinstance(answers, list):
        raise ValueError(f"{filename}: answers는 배열이어야 합니다.")

    gold_id_set = set(gold_ids)
    by_qid: dict[str, dict] = {}
    issues: list[str] = []

    for item in answers:
        if not isinstance(item, dict):
            issues.append("객체가 아닌 답변 항목")
            continue
        qid = item.get("qid")
        if not isinstance(qid, str) or qid not in gold_id_set:
            issues.append(f"알 수 없는 qid={qid!r}")
            continue
        if qid in by_qid:
            issues.append(f"{qid} 중복(첫 항목 사용)")
            continue

        answer = item.get("answer")
        if not isinstance(answer, str):
            issues.append(f"{qid} answer 타입 이상")
            answer = ""

        raw = item.get("retrieved")
        raw = raw if isinstance(raw, list) else []
        normalized = [e for e in map(coerce_evidence, raw) if e is not None]
        if len(raw) != len(normalized) or not 1 <= len(raw) <= MAX_RETRIEVED:
            issues.append(f"{qid} retrieved {len(raw)}개 → {len(normalized[:MAX_RETRIEVED])}개")
        by_qid[qid] = {"qid": qid, "answer": answer.strip(),
                       "retrieved": normalized[:MAX_RETRIEVED]}

    for qid in gold_ids:
        if qid not in by_qid:
            issues.append(f"{qid} 누락 → 무응답 처리")
            by_qid[qid] = {"qid": qid, "answer": "", "retrieved": []}

    if issues:
        print(f"[{filename}] 계약 위반 {len(issues)}건: {issues}")
    return by_qid


def has_unjudgeable_foreign_output(answer: str) -> bool:
    """한국어 없이 중국어 한자로만 구성된 답변을 판정 불가로 식별합니다."""
    hangul_count = len(re.findall(r"[가-힣]", answer))
    han_count = len(re.findall(r"[\u4e00-\u9fff]", answer))
    return hangul_count == 0 and han_count >= 2


# =====================================================================================
# 4. MRR 10점과 키팩트 F1 20점
# =====================================================================================

_kiwi = Kiwi()
_KEEP_TAGS = {"NNG", "NNP", "NNB", "SL", "SH", "SN", "VV", "VA", "XR", "MAG"}


def tokenize_for_f1(text: str) -> list[str]:
    """유니코드 정규화 후 한국어 내용어를 소문자 토큰으로 반환합니다."""
    normalized = unicodedata.normalize("NFKC", text).lower()
    return [token.form.lower() for token in _kiwi.tokenize(normalized) if token.tag in _KEEP_TAGS]


def reciprocal_rank(gold: dict, candidate: dict) -> float:
    """첫 정답 문서·조항의 reciprocal rank를 계산합니다."""
    gold_pairs = {
        (str(item["doc"]).strip(), normalize_article(item["article"]))
        for item in gold["gold_articles"]
    }
    for rank, evidence in enumerate(candidate["retrieved"], start=1):
        if (evidence[0], normalize_article(evidence[1])) in gold_pairs:
            return 1.0 / rank
    return 0.0


def keyfact_f1(gold: dict, candidate: dict) -> float:
    """모든 key_facts와 후보 답변의 내용어 다중집합 F1을 계산합니다."""
    gold_tokens = tokenize_for_f1(" ".join(str(value) for value in gold["key_facts"]))
    answer_tokens = tokenize_for_f1(candidate["answer"])
    if not gold_tokens or not answer_tokens:
        return 0.0
    gold_counts = Counter(gold_tokens)
    answer_counts = Counter(answer_tokens)
    overlap = sum((gold_counts & answer_counts).values())
    precision = overlap / len(answer_tokens)
    recall = overlap / len(gold_tokens)
    return 2.0 * precision * recall / (precision + recall) if precision + recall else 0.0


# =====================================================================================
# 5. 약관 4종 로딩과 관련 조항 추출
# =====================================================================================

TERM_SUFFIXES = {".txt", ".pdf"}
ARTICLE_HEADING_RE = re.compile(r"제\s*(\d+)\s*조(?:\s*\([^\n)]*\))?")


def normalize_document_name(value: str) -> str:
    """유니코드·공백·문장부호·날짜 차이를 제거해 약관 문서명을 비교합니다."""
    normalized = unicodedata.normalize("NFKC", value).lower()
    normalized = re.sub(r"_?\d{8}$", "", Path(normalized).stem)
    return re.sub(r"[^0-9a-z가-힣]", "", normalized)


def read_term_text(path: Path) -> str:
    """UTF-8 TXT 또는 텍스트 기반 PDF에서 약관 원문을 읽습니다."""
    if path.suffix.lower() == ".txt":
        text = path.read_text(encoding="utf-8")
    elif path.suffix.lower() == ".pdf":
        text = "\n".join((page.extract_text() or "") for page in PdfReader(str(path)).pages)
    else:
        raise ValueError(f"지원하지 않는 약관 파일 형식입니다: {path.name}")
    text = unicodedata.normalize("NFKC", text).replace("\x00", "")
    text = re.sub(r"[ \t]+", " ", text)
    if not text.strip():
        raise ValueError(f"약관 텍스트를 읽지 못했습니다: {path.name}")
    return text.strip()


def split_term_articles(text: str) -> dict[int, str]:
    """약관 원문을 제N조 제목 경계로 나눠 조번호별 본문을 반환합니다."""
    matches = list(ARTICLE_HEADING_RE.finditer(text))
    articles: dict[int, str] = {}
    for index, match in enumerate(matches):
        article = int(match.group(1))
        end = matches[index + 1].start() if index + 1 < len(matches) else len(text)
        section = re.sub(r"\s+", " ", text[match.start():end]).strip()
        if section and article not in articles:
            articles[article] = section
    return articles


def load_terms_index(terms_dir: Path, gold_questions: list[dict] | None = None) -> dict[tuple[str, int], str]:
    """약관 4종을 골드 문서명과 연결하고 모든 조항을 색인합니다."""
    if not terms_dir.is_dir():
        raise FileNotFoundError(f"약관 폴더가 없습니다: {terms_dir}")
    files = sorted(path for path in terms_dir.iterdir() if path.is_file() and path.suffix.lower() in TERM_SUFFIXES)
    if len(files) != 4:
        raise ValueError(f"{terms_dir}에는 TXT/PDF 약관이 정확히 4개 있어야 하지만 {len(files)}개입니다: {[p.name for p in files]}")

    gold_docs = sorted({str(item["doc"]).strip() for gold in (gold_questions or []) for item in gold["gold_articles"]})
    index: dict[tuple[str, int], str] = {}
    matched_docs: set[str] = set()
    for path in files:
        file_name = normalize_document_name(path.name)
        matches = [doc for doc in gold_docs if normalize_document_name(doc) in file_name or file_name in normalize_document_name(doc)]
        # 해당 문서가 이번 question에 등장하지 않아도 약관 4종 전체를 정상 색인합니다.
        fallback_doc = re.sub(r"_?\d{8}$", "", unicodedata.normalize("NFKC", path.stem)).strip()
        doc = max(matches, key=lambda value: len(normalize_document_name(value))) if matches else fallback_doc
        if doc in matched_docs:
            raise ValueError(f"약관 문서가 중복 연결되었습니다: {doc}")
        matched_docs.add(doc)
        articles = split_term_articles(read_term_text(path))
        if not articles:
            raise ValueError(f"{path.name}: 제N조 형식의 조항을 찾지 못했습니다.")
        for article, text in articles.items():
            index[(doc, article)] = text[:MAX_TERM_CHARS_PER_ARTICLE]

    return index


def terms_fingerprint(terms_dir: Path) -> str:
    """약관 파일명·크기·수정 시각으로 캐시 무효화용 지문을 만듭니다."""
    import hashlib

    files = sorted(path for path in terms_dir.iterdir() if path.is_file() and path.suffix.lower() in TERM_SUFFIXES) if terms_dir.is_dir() else []
    signature = [(path.name, path.stat().st_size, path.stat().st_mtime_ns) for path in files]
    return hashlib.sha256(json.dumps(signature, ensure_ascii=False).encode("utf-8")).hexdigest()


def load_terms_index_cached(terms_dir: Path, cache_path: Path) -> dict[tuple[str, int], str]:
    """약관이 바뀌지 않았으면 JSON 캐시를 사용하고 아니면 PDF/TXT를 다시 색인합니다."""
    fingerprint = terms_fingerprint(terms_dir)
    if cache_path.is_file():
        try:
            cached = load_json(cache_path)
            if cached.get("fingerprint") == fingerprint and isinstance(cached.get("articles"), list):
                index = {(item["doc"], int(item["article"])): item["text"] for item in cached["articles"]}
                print(f"[약관 캐시] {cache_path.name} · {len(index)}개 조항")
                return index
        except Exception:
            pass
    index = load_terms_index(terms_dir)
    payload = {"fingerprint": fingerprint, "articles": [{"doc": doc, "article": article, "text": text} for (doc, article), text in sorted(index.items())]}
    save_json_atomic(cache_path, payload)
    print(f"[약관 색인] 4개 파일 · {len(index)}개 조항 · 캐시 저장 완료")
    return index


def validate_terms_for_gold(terms_index: dict[tuple[str, int], str], gold_questions: list[dict]) -> None:
    """question의 모든 정답 문서·조항이 약관 색인에 존재하는지 확인합니다."""
    required = {(str(item["doc"]).strip(), normalize_article(item["article"])) for gold in gold_questions for item in gold["gold_articles"]}
    missing = sorted(required - set(terms_index))
    if missing:
        raise ValueError(f"약관 원문에서 정답 조항을 찾지 못했습니다: {missing}")


def select_term_evidence(gold: dict, candidate: dict, terms_index: dict[tuple[str, int], str]) -> list[dict]:
    """정답 조항과 후보 retrieved 조항의 원문을 중복 없이 선택합니다."""
    pairs = [(str(item["doc"]).strip(), normalize_article(item["article"])) for item in gold["gold_articles"]]
    pairs.extend((str(item[0]).strip(), normalize_article(item[1])) for item in candidate["retrieved"])
    selected: list[dict] = []
    seen: set[tuple[str, int]] = set()
    for pair in pairs:
        if pair in seen or pair not in terms_index:
            continue
        seen.add(pair)
        selected.append({"doc": pair[0], "article": pair[1], "text": terms_index[pair]})
    return selected


# =====================================================================================
# 6. Gemini 3.5 Flash LLM 평가 70점
# =====================================================================================

SYSTEM_INSTRUCTION = """당신은 한국어 약관 RAG 답변의 엄격하고 일관된 평가자입니다.
주어진 질문, 정답 근거 조항, 핵심 사실, 관련 약관 원문, 후보 답변과 후보 retrieved만 평가하세요.
judgeable은 답변 내용을 정상적으로 판별할 수 있으면 true, 출력 손상·해독 불가 등으로 판별할 수 없으면 false입니다.
단순히 틀렸거나 정답이 없거나 핵심 사실이 누락된 답변은 판별 가능한 오답이므로 judgeable=true로 두고 0점까지 감점하세요.
외부 지식이나 추측을 사용하지 마세요. 각 축은 0점부터 10점까지 실수로 판정하세요.

축 정의:
- correctness: 질문에 대한 결론·수치·조건·예외·법적 효과가 정답 기준과 정확히 일치하는가. 원문에 없는 주장이나 정반대 결론은 크게 감점한다.
- grounding: 답변이 관련 약관 원문과 후보 retrieved에 의해 실제로 뒷받침되며 근거와 반대되지 않는가. 환각·잘못된 조항 귀속은 크게 감점한다.
- completeness: 질문과 key_facts가 요구하는 핵심 요소를 빠짐없이 포함하는가.
- clarity: 한국어가 자연스럽고 직접적이며 모호하거나 모순되지 않는가. 같은 내용의 반복과 불필요한 장황함은 여기서 감점한다.

짧다는 이유만으로 감점하지 말고 필요한 핵심이 빠졌을 때만 완결성을 감점하세요.
길다는 이유만으로 감점하지 말고 무관한 내용이나 반복이 실제로 있을 때만 감점하세요.
자연스러운 문장이라는 이유만으로 높은 점수를 주지 말고 키팩트 포함 여부와 잘못 추가된 주장을 먼저 확인하세요.
점수 기준: 10=오류·누락 없음, 7=결론은 맞지만 핵심 조건 하나 누락, 4=일부 사실만 맞고 중요한 오류 존재, 1=대부분 불일치, 0=정반대·무관·전부 환각. 필요하면 사이 점수를 사용할 수 있습니다.
retrieved 순위 자체는 MRR에서 별도 계산하므로 grounding에서는 근거 일치성과 답변의 근거성을 판단하세요.
reason에는 가장 중요한 감점 또는 만점 이유만 간결하게 작성하세요."""


def get_api_key() -> str:
    """환경변수 또는 Colab Secret에서 API 키를 읽으며 코드에는 키를 저장하지 않습니다."""
    key = os.environ.get("GEMINI_API_KEY", "").strip()
    if key:
        return key
    try:
        from google.colab import userdata

        key = (userdata.get("GEMINI_API_KEY") or "").strip()
    except Exception:
        key = ""
    if not key:
        raise RuntimeError(
            "GEMINI_API_KEY가 없습니다. Colab Secret에 GEMINI_API_KEY를 등록하거나 환경변수로 설정하세요."
        )
    return key


def build_llm_prompt(gold: dict, candidate: dict, term_evidence: list[dict]) -> str:
    """한 문항 평가에 필요한 정보만 압축 JSON으로 구성합니다."""
    payload = {
        "question": gold["question"],
        "gold_articles": gold["gold_articles"],
        "key_facts": gold["key_facts"],
        "relevant_terms": term_evidence,
        "candidate_retrieved": candidate["retrieved"],
        "candidate_answer": candidate["answer"],
    }
    return json.dumps(payload, ensure_ascii=False, separators=(",", ":"))


def judge_with_gemini(client: genai.Client, gold: dict, candidate: dict, term_evidence: list[dict]) -> LLMJudgment:
    """Gemini 구조화 출력을 호출하고 일시적 실패만 지수 백오프로 재시도합니다."""
    if not candidate["answer"].strip():
        return LLMJudgment(judgeable=True, correctness=0, grounding=0, completeness=0, clarity=0, reason="답변이 비어 있어 정답을 포함하지 않습니다.")
    if has_unjudgeable_foreign_output(candidate["answer"]):
        return LLMJudgment(judgeable=True, correctness=0, grounding=0,
                           completeness=0, clarity=0,
                           reason="한국어 답변이 아니어서 정답을 전달하지 못했습니다.")
    prompt = build_llm_prompt(gold, candidate, term_evidence)
    last_error: Exception | None = None

    for attempt in range(MAX_API_RETRIES):
        try:
            response = client.models.generate_content(
                model=GEMINI_MODEL,
                contents=prompt,
                config=types.GenerateContentConfig(
                    system_instruction=SYSTEM_INSTRUCTION,
                    thinking_config=types.ThinkingConfig(thinking_level="low"),
                    max_output_tokens=512,
                    response_mime_type="application/json",
                    response_schema=LLMJudgment,
                ),
            )
            judgment = response.parsed if isinstance(response.parsed, LLMJudgment) else LLMJudgment.model_validate_json(response.text)
            if not judgment.judgeable:
                raise ValueError(f"Gemini가 판정 불가로 분류했습니다: {judgment.reason}")
            return judgment
        except Exception as error:
            last_error = error
            if attempt + 1 < MAX_API_RETRIES:
                time.sleep(RETRY_BASE_SECONDS * (2**attempt))

    raise RuntimeError(f"Gemini 평가가 {MAX_API_RETRIES}회 실패했습니다: {last_error}")


def llm_score_0_100(judgment: LLMJudgment) -> float:
    """Gemini 축별 0~10 점수를 사전 고정 가중치로 0~100점에 변환합니다."""
    weighted_0_10 = sum(
        getattr(judgment, axis) * weight for axis, weight in LLM_AXIS_WEIGHTS.items()
    )
    return max(0.0, min(100.0, weighted_0_10 * 10.0))


# =====================================================================================
# 7. 체크포인트, 후보 평가와 최종 출력
# =====================================================================================

def load_checkpoint(path: Path) -> dict:
    """이전 성공 문항을 재사용해 불필요한 API 재호출을 막습니다."""
    if not path.is_file():
        return {"model": GEMINI_MODEL, "items": {}}
    data = load_json(path)
    if data.get("model") != GEMINI_MODEL or not isinstance(data.get("items"), dict):
        return {"model": GEMINI_MODEL, "items": {}}
    return data


def save_json_atomic(path: Path, data: dict) -> None:
    """임시 파일을 교체하는 방식으로 중단 중 JSON 손상을 방지합니다."""
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as file:
        json.dump(data, file, ensure_ascii=False, indent=2, allow_nan=False)
    temporary.replace(path)


def checkpoint_key(blind_number: int, qid: str, candidate: dict, term_evidence: list[dict]) -> str:
    """후보 답변과 관련 약관 원문이 같을 때만 체크포인트를 재사용합니다."""
    import hashlib

    content = json.dumps({"candidate": candidate, "terms": term_evidence}, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    digest = hashlib.sha256(content.encode("utf-8")).hexdigest()[:16]
    return f"{blind_number}:{qid}:{digest}"


def evaluate_candidate(
    client: genai.Client,
    blind_number: int,
    gold_questions: list[dict],
    answers_by_qid: dict[str, dict],
    terms_index: dict[tuple[str, int], str],
    checkpoint: dict,
) -> tuple[float | None, Literal["completed", "partial", "failed"], list[dict]]:
    """한 익명 후보의 모든 문항을 순차 평가하고 0~100 총점을 계산합니다."""
    question_results: list[dict] = []
    failures = 0

    for index, gold in enumerate(gold_questions, start=1):
        qid = gold["id"]
        candidate = answers_by_qid[qid]
        mrr = reciprocal_rank(gold, candidate)
        f1 = keyfact_f1(gold, candidate)
        term_evidence = select_term_evidence(gold, candidate, terms_index)
        key = checkpoint_key(blind_number, qid, candidate, term_evidence)

        try:
            cached = checkpoint["items"].get(key)
            if cached is not None:
                judgment = LLMJudgment.model_validate(cached["judgment"])
            else:
                judgment = judge_with_gemini(client, gold, candidate, term_evidence)
                checkpoint["items"][key] = {
                    "blind_number": blind_number,
                    "qid": qid,
                    "judgment": judgment.model_dump(),
                }
                save_json_atomic(CHECKPOINT_PATH, checkpoint)

            llm = llm_score_0_100(judgment)
            question_results.append(
                {
                    "qid": qid,
                    "mrr": mrr,
                    "f1": f1,
                    "llm": llm,
                    "reason": judgment.reason,
                }
            )
            print(f"[blind_{blind_number}] {index}/{len(gold_questions)} {qid} 완료")
        except Exception as error:
            failures += 1
            question_results.append(
                {"qid": qid, "mrr": mrr, "f1": f1, "llm": None, "error": str(error)}
            )
            print(f"[blind_{blind_number}] {qid} 실패: {error}")

    # 하나라도 판정값이 null이면 후보 총점을 신뢰할 수 없으므로 failed/null로 처리합니다.
    if failures > 0 or any(item["llm"] is None for item in question_results):
        return None, "failed", question_results

    mean_mrr = sum(item["mrr"] for item in question_results) / len(question_results)
    mean_f1 = sum(item["f1"] for item in question_results) / len(question_results)
    mean_llm = sum(item["llm"] for item in question_results) / len(question_results)
    total = MRR_WEIGHT * mean_mrr + F1_WEIGHT * mean_f1 + (LLM_WEIGHT / 100.0) * mean_llm
    total = max(0.0, min(100.0, float(total)))
    return total, "completed", question_results


def validate_final_output(output: dict, expected_count: int) -> None:
    """eval_example.json 계약과 최종 제출 유효 조건을 검사합니다."""
    if set(output) != {"results"} or not isinstance(output["results"], list):
        raise ValueError("최상위에는 results 배열만 있어야 합니다.")
    if len(output["results"]) != expected_count:
        raise ValueError(f"results는 정확히 {expected_count}개여야 합니다.")

    seen: set[str] = set()
    for item in output["results"]:
        if set(item) != {"blind_id", "total", "status"}:
            raise ValueError("각 결과에는 blind_id, total, status만 있어야 합니다.")
        blind_id = item["blind_id"]
        if not isinstance(blind_id, str) or re.fullmatch(r"BLIND\d{2}", blind_id) is None:
            raise ValueError(f"blind_id 형식이 잘못되었습니다: {blind_id!r}")
        if blind_id in seen:
            raise ValueError(f"blind_id가 중복되었습니다: {blind_id}")
        seen.add(blind_id)
        status = item["status"]
        if status not in {"completed", "partial", "failed"}:
            raise ValueError(f"status가 잘못되었습니다: {status!r}")
        total = item["total"]
        if status == "failed":
            if total is not None:
                raise ValueError("failed의 total은 null이어야 합니다.")
        elif not isinstance(total, (int, float)) or isinstance(total, bool) or not 0 <= total <= 100:
            raise ValueError("completed/partial의 total은 0~100 숫자여야 합니다.")


def run_evaluation(gold_path: Path, gold_questions: list[dict], terms_index: dict[tuple[str, int], str], candidate_files: list[tuple[int, Path, dict]]) -> dict:
    """검증 완료 입력으로 Gemini 평가를 실행하고 eval_10.json을 저장합니다."""
    gold_ids = [item["id"] for item in gold_questions]

    print(f"[준비] question={gold_path}, 약관조항={len(terms_index)}, 문항={len(gold_questions)}, 후보={len(candidate_files)}")
    client = genai.Client(api_key=get_api_key())
    checkpoint = load_checkpoint(CHECKPOINT_PATH)
    output_results: list[dict] = []

    for blind_number, path, candidate_data in candidate_files:
        blind_id = f"BLIND{blind_number:02d}"
        print(f"[{blind_id}] 입력={path.name}")
        try:
            candidate = validate_candidate(candidate_data, gold_ids, path.name)
            total, status, _details = evaluate_candidate(
                client, blind_number, gold_questions, candidate, terms_index, checkpoint
            )
        except Exception as error:
            total, status = None, "failed"
            print(f"[{path.name}] 후보 전체 실패: {error}")

        output_results.append({"blind_id": blind_id, "total": total, "status": status})

    output = {"results": output_results}
    validate_final_output(output, len(candidate_files))
    save_json_atomic(OUTPUT_PATH, output)

    print(f"[완료] {OUTPUT_PATH}")
    if not all(item["status"] == "completed" for item in output_results):
        print("[주의] partial/failed 후보가 있어 이 파일은 운영진 순위 산정에서 제외됩니다.")
    return output


# 실제 실행 호출은 4번 셀에 있습니다.

# =====================================================================================
# 셀 1 실행: 약관 4종 로드 및 캐싱
# =====================================================================================
TERMS_INDEX = load_terms_index_cached(TERMS_DIR, TERMS_CACHE_PATH)
print(f"[셀 1 완료] terms={TERMS_DIR} · 약관 조항={len(TERMS_INDEX)}")


[약관 캐시] .terms_index_cache.json · 78개 조항
[셀 1 완료] terms=/content/terms · 약관 조항=78


In [8]:
# =====================================================================================
# 셀 2: question 파일 확인 및 10~50문항 동적 로드
# =====================================================================================
# /content/question에 JSON이 없거나 두 개 이상이면 여기서 명확한 오류를 냅니다.
GOLD_PATH = find_gold_path(QUESTION_DIR)
GOLD_QUESTIONS = validate_gold(load_json(GOLD_PATH))
validate_terms_for_gold(TERMS_INDEX, GOLD_QUESTIONS)
GOLD_IDS = [item["id"] for item in GOLD_QUESTIONS]
print(f"[셀 2 완료] question={GOLD_PATH.name} · 문항={len(GOLD_QUESTIONS)} · 정답 조항 검증 완료")


[셀 2 완료] question=gold_questions_private30.json · 문항=30 · 정답 조항 검증 완료


In [9]:
# =====================================================================================
# 셀 3: blind 후보 파일 확인 및 입력 스키마 사전검증
# =====================================================================================
# /content/blind에 후보가 없거나 BLIND01~BLIND05가 누락·중복되면 여기서 오류를 냅니다.
CANDIDATE_FILES = discover_candidate_files(BLIND_DIR)
for blind_number, path, candidate_data in CANDIDATE_FILES:
    validate_candidate(candidate_data, GOLD_IDS, path.name)
    print(f"[BLIND{blind_number:02d}] {path.name} · 답변={len(candidate_data['answers'])}개 · 검증 완료")
print(f"[셀 3 완료] 후보={len(CANDIDATE_FILES)}개")


[BLIND01] answers_private_BLIND01.json · 답변=30개 · 검증 완료
[answers_private_BLIND02.json] 계약 위반 1건: ['F10 retrieved 0개 → 0개']
[BLIND02] answers_private_BLIND02.json · 답변=30개 · 검증 완료
[BLIND03] answers_private_BLIND03.json · 답변=30개 · 검증 완료
[BLIND04] answers_private_BLIND04.json · 답변=30개 · 검증 완료
[BLIND05] answers_private_BLIND05.json · 답변=30개 · 검증 완료
[셀 3 완료] 후보=5개


In [11]:
# =====================================================================================
# 셀 4: 실제 Gemini 평가 실행 및 eval_10.json 생성
# =====================================================================================
# 이 셀부터 Gemini API 비용이 발생합니다. 셀 1~3이 모두 성공한 뒤 실행하세요.
EVAL_OUTPUT = run_evaluation(GOLD_PATH, GOLD_QUESTIONS, TERMS_INDEX, CANDIDATE_FILES)
print(json.dumps(EVAL_OUTPUT, ensure_ascii=False, indent=2))


[준비] question=/content/question/gold_questions_private30.json, 약관조항=78, 문항=30, 후보=5
[BLIND01] 입력=answers_private_BLIND01.json
[blind_1] 1/30 F01 완료
[blind_1] 2/30 F02 완료
[blind_1] 3/30 F03 완료
[blind_1] 4/30 F04 완료
[blind_1] 5/30 F05 완료
[blind_1] 6/30 F06 완료
[blind_1] 7/30 F07 완료
[blind_1] 8/30 F08 완료
[blind_1] 9/30 F09 완료
[blind_1] 10/30 F10 완료
[blind_1] 11/30 F11 완료
[blind_1] 12/30 F12 완료
[blind_1] 13/30 F13 완료
[blind_1] 14/30 F14 완료
[blind_1] 15/30 F15 완료
[blind_1] 16/30 F16 완료
[blind_1] 17/30 F17 완료
[blind_1] 18/30 F18 완료
[blind_1] 19/30 F19 완료
[blind_1] 20/30 F20 완료
[blind_1] 21/30 F21 완료
[blind_1] 22/30 F22 완료
[blind_1] 23/30 F23 완료
[blind_1] 24/30 F24 완료
[blind_1] 25/30 F25 완료
[blind_1] 26/30 F26 완료
[blind_1] 27/30 F27 완료
[blind_1] 28/30 F28 완료
[blind_1] 29/30 F29 완료
[blind_1] 30/30 F30 완료
[BLIND02] 입력=answers_private_BLIND02.json
[answers_private_BLIND02.json] 계약 위반 1건: ['F10 retrieved 0개 → 0개']
[blind_2] 1/30 F01 완료
[blind_2] 2/30 F02 완료
[blind_2] 3/30 F03 완료
[blind_2] 4/30 F04